# UMBC Men’s Soccer Predictive Analytics

## Simulation Notebook: Simple vs Better Approaches

This notebook builds **two Monte Carlo simulation approaches** for UMBC men’s soccer:

### A) Simple Approach (Baseline)
- Uses **historical context averages** to estimate future match inputs:
  - UMBC shots on goal
  - UMBC shot accuracy
  - Opponent shots on goal
- Uses the existing goal models to generate expected goals (λ)
- Simulates scorelines with Poisson goal scoring

### B) Better Approach (Upgraded)
Uses an **input modeling layer** instead of averages.

We implement two versions:
- **Better (Deterministic Inputs):** uses predicted input means
- **Better++ (Two-Stage Stochastic):** simulates match inputs first, then simulates goals

The goal is to keep everything interpretable and soccer-focused.



In [2]:
# ===============================
# 1) Imports
# ===============================

import pandas as pd
import numpy as np

import statsmodels.api as sm
import statsmodels.formula.api as smf

from pathlib import Path




# 2) Load and Combine Match Data

Expected CSV files:
- team_match_2023.csv
- team_match_2024.csv
- team_match_2025.csv

Confirmed columns:
- season, date, home_away, opponent, result,
- goals_for, goals_against, attendance,
- shots, shots_on_goal, yellow, red,
- opp_shots, opp_shots_on_goal


In [3]:
# ===============================
# 2) Load Data
# ===============================

data_dir = Path(
    r"C:/Users/jdrakes1/Box/AAA-Share/Jervon's Work/Sports Predictive Analytics"
)

df23 = pd.read_csv(data_dir / "team_match_2023.csv")
df24 = pd.read_csv(data_dir / "team_match_2024.csv")
df25 = pd.read_csv(data_dir / "team_match_2025.csv")

df = pd.concat([df23, df24, df25], ignore_index=True)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)


# 3) Feature Engineering (Schema Adapter)

Your raw data does not include:
- home (binary)
- shot_accuracy
- is_conference
- opp_strength
- conf_x_opp

This section creates all required features.

## America East Opponents (Confirmed spellings)
- Ualbany
- Binghamton
- Bryant
- Maine
- UMass Lowell
- New Hampshire
- NJIT
- Vermont


In [4]:
# ===============================
# 3) Feature Engineering
# ===============================

# Helper: normalize opponent strings
def norm_opp(x):
    return str(x).strip()

df["opponent_norm"] = df["opponent"].map(norm_opp)

# home indicator from home_away
ha = df["home_away"].astype(str).str.strip().str.lower()
df["home"] = np.where(ha.str.startswith("h"), 1,
              np.where(ha.str.startswith("a"), 0, np.nan))

# If you ever have unmapped values, inspect:
# if df["home"].isna().any():
#     print("Unmapped home_away values:", df.loc[df["home"].isna(), "home_away"].unique())

df["home"] = df["home"].fillna(0).astype(int)

# shot accuracy
df["shot_accuracy"] = np.where(df["shots"] > 0, df["shots_on_goal"] / df["shots"], 0.0)

# conference teams
AMERICA_EAST = {
    "Ualbany",
    "Binghamton",
    "Bryant",
    "Maine",
    "UMass Lowell",
    "New Hampshire",
    "NJIT",
    "Vermont",
}
AMERICA_EAST_NORM = {norm_opp(x) for x in AMERICA_EAST}

df["is_conference"] = df["opponent_norm"].isin(AMERICA_EAST_NORM).astype(int)

# goal difference (for opp_strength)
df["gd"] = df["goals_for"] - df["goals_against"]

# leak-free opponent strength: prior expanding mean GD vs that opponent (shifted)
df["opp_strength"] = np.nan
for opp, g in df.groupby("opponent_norm", sort=False):
    g = g.sort_values("date")
    prior_strength = g["gd"].expanding().mean().shift(1)
    df.loc[g.index, "opp_strength"] = prior_strength

df["opp_strength"] = df["opp_strength"].fillna(df["gd"].mean())

# interaction term
df["conf_x_opp"] = df["is_conference"] * df["opp_strength"]


# 4) Goal Models (Poisson)

I fit:
- Attack model: goals_for
- Defense model: goals_against

These produce:
- xG_ctx (expected goals for)
- xGA_ctx (expected goals against)
- xGD = xG_ctx − xGA_ctx


In [6]:
# ===============================
# 4) Goal Models (Poisson GLM)
# ===============================

attack_formula = """
goals_for ~ shots_on_goal + shot_accuracy + home +
opp_strength + is_conference + conf_x_opp
"""

attack_model = smf.glm(
    formula=attack_formula,
    data=df,
    family=sm.families.Poisson()
).fit()

defense_formula = """
goals_against ~ opp_shots_on_goal + home +
opp_strength + is_conference + conf_x_opp
"""

defense_model = smf.glm(
    formula=defense_formula,
    data=df,
    family=sm.families.Poisson()
).fit()

df["xG_ctx"] = attack_model.predict(df)
df["xGA_ctx"] = defense_model.predict(df)
df["xGD"] = df["xG_ctx"] - df["xGA_ctx"]


# 5) Input Models for the Better Approach

I model match inputs directly:

- UMBC shots_on_goal (SOG_for) → Negative Binomial
- opponent shots_on_goal (OppSOG) → Negative Binomial
- shot accuracy → Binomial logistic model (successes out of shots)

These models allow future match inputs to respond to:
- opponent strength
- home/away
- conference context


In [7]:
# ===============================
# 5.1) SOG FOR (Negative Binomial)
# ===============================

sog_for_formula = """
shots_on_goal ~ home + opp_strength + is_conference + conf_x_opp
"""

sog_for_model = smf.glm(
    formula=sog_for_formula,
    data=df,
    family=sm.families.NegativeBinomial()
).fit()


C:\Users\jdrakes1\AppData\Local\anaconda3\envs\soccer\lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [8]:
# ===============================
# 5.2) Opponent SOG (Negative Binomial)
# ===============================

sog_against_formula = """
opp_shots_on_goal ~ home + opp_strength + is_conference + conf_x_opp
"""

sog_against_model = smf.glm(
    formula=sog_against_formula,
    data=df,
    family=sm.families.NegativeBinomial()
).fit()


C:\Users\jdrakes1\AppData\Local\anaconda3\envs\soccer\lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [9]:
# ===============================
# 5.3) Shot Accuracy (Binomial Logistic)
# ===============================

df_acc = df[df["shots"] >= 1].copy()
df_acc["acc_rate"] = df_acc["shots_on_goal"] / df_acc["shots"]

acc_formula = """
acc_rate ~ home + opp_strength + is_conference + conf_x_opp
"""

acc_model = smf.glm(
    formula=acc_formula,
    data=df_acc,
    family=sm.families.Binomial(),
    freq_weights=df_acc["shots"]
).fit()


# 6) Simulation Utilities

## Negative Binomial Sampling
We simulate NB counts using a Gamma–Poisson mixture:
Var(Y) = μ + αμ²

We estimate α using a method-of-moments approximation.

## Accuracy Sampling
We add realistic match-to-match variation by sampling:
p* ~ Beta(centered at predicted p)


In [10]:
# ===============================
# 6) Helpers
# ===============================

def estimate_alpha_mom(y: pd.Series) -> float:
    mu = float(y.mean())
    var = float(y.var(ddof=1))
    if mu <= 0:
        return 1e-8
    alpha = (var - mu) / (mu**2)
    return float(max(alpha, 1e-8))

def rnegbin_gamma_poisson(rng, mu, alpha, size=1):
    mu = np.maximum(mu, 1e-12)
    alpha = np.maximum(alpha, 1e-12)
    shape = 1.0 / alpha
    scale = mu * alpha
    lam = rng.gamma(shape=shape, scale=scale, size=size)
    return rng.poisson(lam, size=size)

def beta_from_mean_conc(mean, k):
    mean = float(np.clip(mean, 1e-4, 1-1e-4))
    a = mean * k
    b = (1 - mean) * k
    return a, b

alpha_sog_for = estimate_alpha_mom(df["shots_on_goal"])
alpha_sog_against = estimate_alpha_mom(df["opp_shots_on_goal"])


# 7) Build Future Match Rows (Rematches vs All Opponents)

We create a future dataframe with:
- opponent
- home
- is_conference
- opp_strength (carried forward)
- conf_x_opp


In [11]:
# ===============================
# 7) Future Rows Builder
# ===============================

def build_future_rematches(df, home_value=1):
    future = pd.DataFrame({"opponent": sorted(df["opponent"].unique())})
    future["opponent_norm"] = future["opponent"].map(norm_opp)

    future["home"] = int(home_value)
    future["is_conference"] = future["opponent_norm"].isin(AMERICA_EAST_NORM).astype(int)

    last_strength = (
        df.sort_values("date")
          .groupby("opponent_norm")["opp_strength"]
          .last()
    )

    future["opp_strength"] = (
        future["opponent_norm"].map(last_strength)
        .fillna(df["opp_strength"].mean())
    )

    future["conf_x_opp"] = future["is_conference"] * future["opp_strength"]
    return future

future = build_future_rematches(df, home_value=1)


# 8) Approach A — Simple Simulation (Context Averages)

We estimate future inputs using historical averages grouped by:
- home
- is_conference

Then:
- compute λ_for and λ_against using goal models
- simulate scores with Poisson goals


In [12]:
# ===============================
# 8A) Context averages
# ===============================

context_means = (
    df.groupby(["home", "is_conference"])
      .agg(
          avg_sog_for=("shots_on_goal", "mean"),
          avg_acc=("shot_accuracy", "mean"),
          avg_opp_sog=("opp_shots_on_goal", "mean"),
      )
      .reset_index()
)

future_simple = future.merge(context_means, on=["home", "is_conference"], how="left")

# Fallbacks if a context bucket is missing
future_simple["avg_sog_for"] = future_simple["avg_sog_for"].fillna(df["shots_on_goal"].mean())
future_simple["avg_acc"] = future_simple["avg_acc"].fillna(df["shot_accuracy"].mean())
future_simple["avg_opp_sog"] = future_simple["avg_opp_sog"].fillna(df["opp_shots_on_goal"].mean())


In [16]:
# ===============================
# 8A) Simple simulator
# ===============================

def simulate_simple_row(row, n_sims=20000, seed=7):
    rng = np.random.default_rng(seed)

    xrow = pd.DataFrame([{
        "shots_on_goal": row["avg_sog_for"],
        "shot_accuracy": row["avg_acc"],
        "opp_shots_on_goal": row["avg_opp_sog"],
        "home": row["home"],
        "opp_strength": row["opp_strength"],
        "is_conference": row["is_conference"],
        "conf_x_opp": row["conf_x_opp"],
    }])

    lam_for = float(attack_model.predict(xrow)[0])
    lam_against = float(defense_model.predict(xrow)[0])

    gf = rng.poisson(lam_for, size=n_sims)
    ga = rng.poisson(lam_against, size=n_sims)

    win = np.mean(gf > ga)
    draw = np.mean(gf == ga)
    loss = np.mean(gf < ga)

    return {
        "lam_for": lam_for,
        "lam_against": lam_against,
        "win_prob": float(win),
        "draw_prob": float(draw),
        "loss_prob": float(loss),
        "exp_pts": float(3*win + 1*draw),
        "exp_goal_diff": float(np.mean(gf - ga)),
        "gf_mean": float(np.mean(gf)),
        "ga_mean": float(np.mean(ga)),
        "gf_sd": float(np.std(gf)),
        "ga_sd": float(np.std(ga)),
    }

rows = []
for _, r in future_simple.iterrows():
    out = simulate_simple_row(r.to_dict(), n_sims=20000, seed=7)
    rows.append({**r.to_dict(), **out})

sim_simple = pd.DataFrame(rows).sort_values("exp_pts", ascending=False)
sim_simple.head(10)


,opponent,opponent_norm,home,is_conference,opp_strength,conf_x_opp,avg_sog_for,avg_acc,avg_opp_sog,lam_for,lam_against,win_prob,draw_prob,loss_prob,exp_pts,exp_goal_diff,gf_mean,ga_mean,gf_sd,ga_sd
17,Niagara,Niagara,1,0,4.000000,0.000000,6.52381,0.435819,5.142857,1.699401,0.474339,0.68160,0.22180,0.09660,2.26660,1.24390,1.71265,0.46875,1.311709,0.681780
14,NJIT,NJIT,1,1,1.000000,1.000000,5.30000,0.399463,3.700000,1.504592,0.802517,0.54605,0.25385,0.20010,1.89200,0.71865,1.51485,0.79620,1.228283,0.891272
2,Binghamton,Binghamton,1,1,1.000000,1.000000,5.30000,0.399463,3.700000,1.504592,0.802517,0.54605,0.25385,0.20010,1.89200,0.71865,1.51485,0.79620,1.228283,0.891272
11,Howard,Howard,1,0,1.500000,0.000000,6.52381,0.435819,5.142857,1.572691,0.954442,0.52185,0.25270,0.22545,1.81825,0.63500,1.58485,0.94985,1.257895,0.973054
3,Bryant,Bryant,1,1,0.500000,0.500000,5.30000,0.399463,3.700000,1.322111,0.863934,0.47925,0.27925,0.24150,1.71700,0.47565,1.33075,0.85510,1.148022,0.927580
4,Cornell,Cornell,1,0,1.000000,0.000000,6.52381,0.435819,5.142857,1.548507,1.097695,0.48085,0.25735,0.26180,1.69990,0.46835,1.55990,1.09155,1.246440,1.043201
7,George Mason,George Mason,1,0,1.000000,0.000000,6.52381,0.435819,5.142857,1.548507,1.097695,0.48085,0.25735,0.26180,1.69990,0.46835,1.55990,1.09155,1.246440,1.043201
24,Vermont,Vermont,1,1,0.333333,0.333333,5.30000,0.399463,3.700000,1.266342,0.885434,0.45270,0.29465,0.25265,1.65275,0.39365,1.27145,0.87780,1.123372,0.937746
23,Umass Lowell,Umass Lowell,1,0,0.500000,0.000000,6.52381,0.435819,5.142857,1.524694,1.262450,0.43890,0.24725,0.31385,1.56395,0.27495,1.53510,1.26015,1.237000,1.119898
22,Ualbany,Ualbany,1,1,0.000000,0.000000,5.30000,0.399463,3.700000,1.161762,0.930051,0.41645,0.29695,0.28660,1.54630,0.24505,1.16470,0.91965,1.077903,0.958120


# 9) Approach B — Better Simulation (Input Modeling)

We use match input models to generate:
- predicted UMBC SOG_for
- predicted OppSOG
- predicted accuracy

We implement:

## B1) Better (Deterministic Inputs)
Uses predicted means only.

## B2) Better++ (Two-Stage Stochastic)
Simulates inputs each trial (NB + Beta) then simulates goals.


In [14]:
# ===============================
# 9B1) Better deterministic
# ===============================

def simulate_better_deterministic_row(row, n_sims=20000, seed=7):
    rng = np.random.default_rng(seed)

    base = pd.DataFrame([{
        "home": row["home"],
        "opp_strength": row["opp_strength"],
        "is_conference": row["is_conference"],
        "conf_x_opp": row["conf_x_opp"],
    }])

    pred_sog_for = float(sog_for_model.predict(base)[0])
    pred_opp_sog = float(sog_against_model.predict(base)[0])
    pred_acc = float(np.clip(acc_model.predict(base)[0], 0.01, 0.99))

    xrow = pd.DataFrame([{
        "shots_on_goal": pred_sog_for,
        "shot_accuracy": pred_acc,
        "opp_shots_on_goal": pred_opp_sog,
        "home": row["home"],
        "opp_strength": row["opp_strength"],
        "is_conference": row["is_conference"],
        "conf_x_opp": row["conf_x_opp"],
    }])

    lam_for = float(attack_model.predict(xrow)[0])
    lam_against = float(defense_model.predict(xrow)[0])

    gf = rng.poisson(lam_for, size=n_sims)
    ga = rng.poisson(lam_against, size=n_sims)

    win = np.mean(gf > ga)
    draw = np.mean(gf == ga)
    loss = np.mean(gf < ga)

    return {
        "pred_sog_for": pred_sog_for,
        "pred_opp_sog": pred_opp_sog,
        "pred_acc": pred_acc,
        "lam_for": lam_for,
        "lam_against": lam_against,
        "win_prob": float(win),
        "draw_prob": float(draw),
        "loss_prob": float(loss),
        "exp_pts": float(3*win + 1*draw),
        "exp_goal_diff": float(np.mean(gf - ga)),
        "gf_mean": float(np.mean(gf)),
        "ga_mean": float(np.mean(ga)),
        "gf_sd": float(np.std(gf)),
        "ga_sd": float(np.std(ga)),
    }

rows = []
for _, r in future.iterrows():
    out = simulate_better_deterministic_row(r.to_dict(), n_sims=20000, seed=7)
    rows.append({**r.to_dict(), **out})

sim_better = pd.DataFrame(rows).sort_values("exp_pts", ascending=False)
# sim_better.head(10)


In [18]:
# ===============================
# 9B2) Better++ two-stage stochastic
# ===============================

def simulate_better_two_stage_row(row, n_sims=20000, seed=7, acc_conc_k=60):
    rng = np.random.default_rng(seed)

    base = pd.DataFrame([{
        "home": row["home"],
        "opp_strength": row["opp_strength"],
        "is_conference": row["is_conference"],
        "conf_x_opp": row["conf_x_opp"],
    }])

    # Stage 1: simulate inputs
    mu_sog_for = float(sog_for_model.predict(base)[0])
    mu_opp_sog = float(sog_against_model.predict(base)[0])

    sog_for_sim = rnegbin_gamma_poisson(rng, mu_sog_for, alpha_sog_for, size=n_sims)
    opp_sog_sim = rnegbin_gamma_poisson(rng, mu_opp_sog, alpha_sog_against, size=n_sims)

    p_acc = float(acc_model.predict(base)[0])
    a, b = beta_from_mean_conc(p_acc, acc_conc_k)
    acc_sim = rng.beta(a, b, size=n_sims)

    # Stage 2: conditional goals
    sim_df = pd.DataFrame({
        "shots_on_goal": sog_for_sim,
        "shot_accuracy": acc_sim,
        "opp_shots_on_goal": opp_sog_sim,
        "home": row["home"],
        "opp_strength": row["opp_strength"],
        "is_conference": row["is_conference"],
        "conf_x_opp": row["conf_x_opp"],
    })

    lam_for = attack_model.predict(sim_df).to_numpy()
    lam_against = defense_model.predict(sim_df).to_numpy()

    gf = rng.poisson(lam_for)
    ga = rng.poisson(lam_against)

    win = np.mean(gf > ga)
    draw = np.mean(gf == ga)
    loss = np.mean(gf < ga)

    return {
        "mu_sog_for": mu_sog_for,
        "mu_opp_sog": mu_opp_sog,
        "p_acc": p_acc,
        "lam_for_mean": float(np.mean(lam_for)),
        "lam_against_mean": float(np.mean(lam_against)),
        "win_prob": float(win),
        "draw_prob": float(draw),
        "loss_prob": float(loss),
        "exp_pts": float(3*win + 1*draw),
        "exp_goal_diff": float(np.mean(gf - ga)),
        "gf_mean": float(np.mean(gf)),
        "ga_mean": float(np.mean(ga)),
        "gf_sd": float(np.std(gf)),
        "ga_sd": float(np.std(ga)),
    }

rows = []
for _, r in future.iterrows():
    out = simulate_better_two_stage_row(r.to_dict(), n_sims=20000, seed=7, acc_conc_k=60)
    rows.append({**r.to_dict(), **out})

sim_better2 = pd.DataFrame(rows).sort_values("exp_pts", ascending=False)
sim_better2.head(10)


,opponent,opponent_norm,home,is_conference,opp_strength,conf_x_opp,mu_sog_for,mu_opp_sog,p_acc,lam_for_mean,lam_against_mean,win_prob,draw_prob,loss_prob,exp_pts,exp_goal_diff,gf_mean,ga_mean,gf_sd,ga_sd
17,Niagara,Niagara,1,0,4.000000,0.000000,5.794918,5.538114,0.460457,1.706628,0.562842,0.62570,0.24225,0.13205,2.11935,1.14330,1.70595,0.56265,1.527247,0.815889
11,Howard,Howard,1,0,1.500000,0.000000,6.332940,5.009180,0.446599,1.694253,1.026283,0.51130,0.24115,0.24755,1.77505,0.68290,1.70360,1.02070,1.610263,1.131800
14,NJIT,NJIT,1,1,1.000000,1.000000,4.979155,4.594364,0.372841,1.518957,1.010998,0.48485,0.25580,0.25935,1.71035,0.52150,1.52265,1.00115,1.383180,1.099340
2,Binghamton,Binghamton,1,1,1.000000,1.000000,4.979155,4.594364,0.372841,1.518957,1.010998,0.48485,0.25580,0.25935,1.71035,0.52150,1.52265,1.00115,1.383180,1.099340
4,Cornell,Cornell,1,0,1.000000,0.000000,6.446396,4.909616,0.443837,1.692689,1.157336,0.48775,0.23665,0.27560,1.69990,0.54900,1.69790,1.14890,1.568004,1.223490
7,George Mason,George Mason,1,0,1.000000,0.000000,6.446396,4.909616,0.443837,1.692689,1.157336,0.48775,0.23665,0.27560,1.69990,0.54900,1.69790,1.14890,1.568004,1.223490
23,Umass Lowell,Umass Lowell,1,0,0.500000,0.000000,6.561885,4.812032,0.441078,1.685042,1.308356,0.46055,0.22830,0.31115,1.60995,0.38545,1.69625,1.31080,1.557365,1.307556
3,Bryant,Bryant,1,1,0.500000,0.500000,5.200249,4.378915,0.392631,1.391817,1.044548,0.44620,0.26555,0.28825,1.60415,0.35025,1.39570,1.04545,1.327826,1.130480
24,Vermont,Vermont,1,1,0.333333,0.333333,5.276107,4.309368,0.399311,1.353130,1.057636,0.42825,0.27095,0.30080,1.55570,0.28660,1.34405,1.05745,1.314412,1.132453
13,Maryland,Maryland,1,0,0.192308,0.000000,6.633981,4.752947,0.439382,1.687887,1.412665,0.44015,0.22425,0.33560,1.54470,0.28870,1.69665,1.40795,1.571505,1.352933


# 10) Compare Approaches

We combine the outputs into one table so we can compare:
- expected points
- win probability
- expected goal difference

Across:
- Simple
- Better (deterministic)
- Better++ (two-stage stochastic)


In [20]:
# ===============================
# 10) Comparison table
# ===============================

comp = (
    sim_simple[["opponent","exp_pts","win_prob","draw_prob","loss_prob","exp_goal_diff"]]
      .rename(columns=lambda c: f"simple_{c}" if c != "opponent" else c)
      .merge(
          sim_better[["opponent","exp_pts","win_prob","draw_prob","loss_prob","exp_goal_diff"]]
            .rename(columns=lambda c: f"better_{c}" if c != "opponent" else c),
          on="opponent",
          how="inner"
      )
      .merge(
          sim_better2[["opponent","exp_pts","win_prob","draw_prob","loss_prob","exp_goal_diff"]]
            .rename(columns=lambda c: f"better2_{c}" if c != "opponent" else c),
          on="opponent",
          how="inner"
      )
)

# Example: sort by the most realistic version (two-stage)
comp = comp.sort_values("better2_exp_pts", ascending=False)

comp.head(15)


,opponent,simple_exp_pts,simple_win_prob,simple_draw_prob,simple_loss_prob,simple_exp_goal_diff,better_exp_pts,better_win_prob,better_draw_prob,better_loss_prob,better_exp_goal_diff,better2_exp_pts,better2_win_prob,better2_draw_prob,better2_loss_prob,better2_exp_goal_diff
0,Niagara,2.26660,0.68160,0.22180,0.09660,1.24390,2.17015,0.64105,0.24700,0.11195,1.08835,2.11935,0.62570,0.24225,0.13205,1.14330
3,Howard,1.81825,0.52185,0.25270,0.22545,0.63500,1.81610,0.52265,0.24815,0.22920,0.62880,1.77505,0.51130,0.24115,0.24755,0.68290
2,Binghamton,1.89200,0.54605,0.25385,0.20010,0.71865,1.73480,0.48975,0.26555,0.24470,0.51215,1.71035,0.48485,0.25580,0.25935,0.52150
1,NJIT,1.89200,0.54605,0.25385,0.20010,0.71865,1.73480,0.48975,0.26555,0.24470,0.51215,1.71035,0.48485,0.25580,0.25935,0.52150
5,Cornell,1.69990,0.48085,0.25735,0.26180,0.46835,1.72900,0.49350,0.24850,0.25800,0.50075,1.69990,0.48775,0.23665,0.27560,0.54900
6,George Mason,1.69990,0.48085,0.25735,0.26180,0.46835,1.72900,0.49350,0.24850,0.25800,0.50075,1.69990,0.48775,0.23665,0.27560,0.54900
8,Umass Lowell,1.56395,0.43890,0.24725,0.31385,0.27495,1.61730,0.45475,0.25305,0.29220,0.35705,1.60995,0.46055,0.22830,0.31115,0.38545
4,Bryant,1.71700,0.47925,0.27925,0.24150,0.47565,1.62430,0.44795,0.28045,0.27160,0.35105,1.60415,0.44620,0.26555,0.28825,0.35025
7,Vermont,1.65275,0.45270,0.29465,0.25265,0.39365,1.57710,0.42925,0.28935,0.28140,0.29370,1.55570,0.42825,0.27095,0.30080,0.28660
16,Liberty,1.48150,0.41160,0.24670,0.34170,0.14805,1.55115,0.43465,0.24720,0.31815,0.25750,1.54470,0.44015,0.22425,0.33560,0.28870


# Results Interpretation

## Overall Pattern

The comparison between the **Simple**, **Better**, and **Better++ (Two-Stage)** simulation approaches shows how increasing realism affects predicted match outcomes.

Across most opponents, the **Simple approach produces slightly higher expected points and win probabilities**. This suggests that the baseline model is somewhat optimistic. Because it relies on historical context averages (home vs. away, conference vs. non-conference), it assumes future matches will resemble typical past conditions rather than adjusting fully to the specific opponent.

When moving to the **Better approach**, expected points and win probabilities generally decrease slightly. This happens because key match inputs — such as shots on goal and shot accuracy — are now modeled based on opponent strength and context. The system recognizes that stronger opponents reduce attacking output and increase defensive pressure, leading to more matchup-sensitive and slightly more conservative projections.

The **Better++ (two-stage stochastic) approach** typically produces the most cautious estimates. By first simulating how the match is likely to be played (shot volume, accuracy, defensive pressure) and then simulating goals, this approach introduces realistic variability into the game environment. The result is less overconfidence and outcome distributions that better reflect the natural unpredictability of soccer.

---

## Differentiation Between Opponents

In the Simple model, multiple opponents can receive identical predictions because they fall into the same context bucket (home and conference status). This highlights a limitation of the baseline approach — it does not distinguish between opponents beyond broad categories.

The Better approaches remove this limitation by incorporating opponent strength directly into the input models. As a result, projections become more opponent-specific and better reflect matchup differences.

Stronger opponents tend to show larger reductions in expected points under the Better models, while weaker opponents remain favorable matchups but with slightly reduced confidence. This indicates improved calibration to competitive balance.

---

## Practical Implications

The Simple approach works well as a baseline and proof of concept. It confirms that the modeling and simulation pipeline functions properly and produces reasonable outputs.

However, the Better and Better++ approaches provide:

- More realistic win probabilities  
- Reduced overconfidence in favorable matchups  
- Greater sensitivity to opponent strength  
- Improved representation of match uncertainty  

Overall, the progression from Simple → Better → Better++ demonstrates increasing realism and robustness while maintaining interpretability. The upgraded models are therefore more appropriate for tactical evaluation and forward-looking performance analysis.
